In [9]:
pip install pillow opencv-python pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [10]:
from pathlib import Path
import os
import shutil
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt 

raw_dir = Path('/Users/adult/Downloads/archive/data/quarantined/asl_alphabet_train')
cleaned_dir = Path( './data/cleaned')
quarantine_dir = Path('./data/quarantined')
reports_dir = Path('./reports')

cleaned_dir.mkdir(parents=True,exist_ok=True)
quarantine_dir.mkdir(parents=True,exist_ok=True)
reports_dir.mkdir(parents=True,exist_ok=True)

print(raw_dir.exists())

True


# Clean and Inspect Images

In [11]:
fremoved_files = []
image_records = []

valid_formats = [".jpg", ".jpeg", ".png"]
min_width = 50
min_height = 50

for class_path in raw_dir.iterdir():

    if not class_path.is_dir():
        continue

    class_name = class_path.name
    print("Processing:", class_name)

    cleaned_class_path = cleaned_dir / class_name
    quarantine_class_path = quarantine_dir / class_name

    cleaned_class_path.mkdir(parents=True, exist_ok=True)
    quarantine_class_path.mkdir(parents=True, exist_ok=True)

    for file_path in class_path.iterdir():

        if not file_path.is_file():
            continue

        file_name = file_path.name
        file_ext = file_path.suffix.lower()

        try:
            if file_ext not in valid_formats:
                reason = "Invalid format"
                shutil.move(str(file_path), str(quarantine_class_path / file_name))
                removed_files.append([file_name, class_name, reason])
                continue

            img = Image.open(file_path)
            img.verify()

            img = Image.open(file_path)
            width, height = img.size

            if width < min_width or height < min_height:
                reason = "Image too small"
                shutil.move(str(file_path), str(quarantine_class_path / file_name))
                removed_files.append([file_name, class_name, reason])
                continue

            if img.getbbox() is None:
                reason = "Blank image"
                shutil.move(str(file_path), str(quarantine_class_path / file_name))
                removed_files.append([file_name, class_name, reason])
                continue

            shutil.copy(str(file_path), str(cleaned_class_path / file_name))

            image_records.append([
                file_name,
                class_name,
                width,
                height,
                file_ext
            ])

        except Exception as e:
            reason = f"Corrupted or unreadable image: {e}"
            removed_files.append([file_name, class_name, reason])

print("Image records:", len(image_records))
print("Removed files:", len(removed_files))

Processing: R
Processing: U
Processing: I
Processing: N
Processing: G
Processing: Z
Processing: T
Processing: S
Processing: A
Processing: F
Processing: O
Processing: H
Processing: del
Processing: nothing
Processing: space
Processing: M
Processing: J
Processing: C
Processing: D
Processing: V
Processing: Q
Processing: X
Processing: E
Processing: B
Processing: K
Processing: L
Processing: Y
Processing: P
Processing: W
Image records: 87000


NameError: name 'removed_files' is not defined

# Removed Files Report

In [12]:
removed_df = pd.DataFrame(
    removed_files,
    columns=["asl_alphabet", "class", "reason_removed"])

reports_dir=Path('./reports')
reports_dir.mkdir(parents=True, exist_ok=True)

removed_df.to_csv(reports_dir/"removed_files.csv", index=False)

removed_df.head()

NameError: name 'removed_files' is not defined